In [1]:
# ============================================
# CELL 1 — IMPORTS + PATHS
# ============================================
import json, warnings
from pathlib import Path
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import itertools
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ── INPUT: add 2 datasets vào notebook này ──
# 1. Output của stage 0+2 (chứa stage0_artifacts/)
# 2. Output của stage 3   (chứa stage3_artifacts/)
STAGE0_DIR = Path("/kaggle/input/notebooks/tinybullheaded/stage-0-and-2-clothing-shoes-and-jewelry/stage0_artifacts")
STAGE3_INPUT_DIR = Path("/kaggle/input/notebooks/tinybullheaded/stage-3-clothing-shoes-and-jewelry/stage3_artifacts")
STAGE4_DIR = Path("/kaggle/working/stage4_artifacts")
STAGE4_DIR.mkdir(parents=True, exist_ok=True)

COL_USER   = "reviewerID"
COL_ITEM   = "asin"
COL_RATING = "overall"
COL_DATE   = "review_date"

print("Stage0 input :", STAGE0_DIR)
print("Stage3 input :", STAGE3_INPUT_DIR)
print("Stage4 output:", STAGE4_DIR)

import os
print("\nStage0 files:", os.listdir(STAGE0_DIR))
print("Stage3 files:", os.listdir(STAGE3_INPUT_DIR))

Stage0 input : /kaggle/input/notebooks/tinybullheaded/stage-0-and-2-clothing-shoes-and-jewelry/stage0_artifacts
Stage3 input : /kaggle/input/notebooks/tinybullheaded/stage-3-clothing-shoes-and-jewelry/stage3_artifacts
Stage4 output: /kaggle/working/stage4_artifacts

Stage0 files: ['user_means.parquet', 'train.parquet', 'df_core.parquet', 'item_means.parquet', 'test.parquet']
Stage3 files: ['model_c_tuning.csv', 'model_d_tuning.csv', 'model_b_val_preds_oof.npy', 'model_c_preds.npy', 'model_a_val_preds_oof.npy', 'model_b_preds.npy', 'model_d_val_preds_oof.npy', 'model_b_result.json', 'model_c_result.json', 'model_d_result.json', 'model_a_preds.npy', 'stage3_oof.parquet', 'model_b_tuning.csv', 'model_a_tuning.csv', 'stage3_test_preds.parquet', 'model_d_preds.npy', 'model_b_Q.npy', 'model_c_val_preds_oof.npy', 'stage3_summary.json', 'model_a_result.json', 'model_b_P.npy']


In [2]:
# ============================================
# CELL 2 — LOAD STAGE 0 + BUILD TEST_WARM
# ============================================
train      = pd.read_parquet(STAGE0_DIR / "train.parquet").copy()
test       = pd.read_parquet(STAGE0_DIR / "test.parquet").copy()
user_stats = pd.read_parquet(STAGE0_DIR / "user_means.parquet")
item_stats = pd.read_parquet(STAGE0_DIR / "item_means.parquet")

for df in [train, test]:
    if COL_DATE in df.columns:
        df[COL_DATE] = pd.to_datetime(df[COL_DATE], errors="coerce")

# Mean maps
if COL_USER in user_stats.columns:
    user_mean_map = dict(zip(user_stats[COL_USER], user_stats["user_mean"]))
else:
    user_mean_map = user_stats["user_mean"].to_dict()

if COL_ITEM in item_stats.columns:
    item_mean_map = dict(zip(item_stats[COL_ITEM], item_stats["item_mean"]))
else:
    item_mean_map = item_stats["item_mean"].to_dict()

GLOBAL_MEAN = float(train[COL_RATING].mean())

# Test warm
train_users = set(train[COL_USER].unique())
train_items = set(train[COL_ITEM].unique())
mask_warm   = test[COL_USER].isin(train_users) & test[COL_ITEM].isin(train_items)
test_warm   = test[mask_warm].reset_index(drop=True)

# Helpers
def clip_rating(x):
    return np.clip(np.asarray(x, dtype=np.float32), 1.0, 5.0)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(
        np.asarray(y_true, np.float32),
        clip_rating(y_pred)
    )))

def mae(y_true, y_pred):
    return float(mean_absolute_error(
        np.asarray(y_true, np.float32),
        clip_rating(y_pred)
    ))

def evaluate(y_true, y_pred, name="model"):
    y_true = np.asarray(y_true, np.float32)
    y_pred = clip_rating(y_pred)
    return {
        "model": name,
        "n":    int(len(y_true)),
        "rmse": rmse(y_true, y_pred),
        "mae":  mae(y_true, y_pred),
    }

def evaluate_per_star(y_true, y_pred):
    y_true = np.asarray(y_true, np.float32)
    y_pred = clip_rating(y_pred)
    out = {}
    for s in range(1, 6):
        mask = (np.rint(y_true).astype(int) == s)
        out[f"star_{s}_n"]    = int(mask.sum())
        out[f"star_{s}_rmse"] = rmse(y_true[mask], y_pred[mask]) if mask.sum() else None
        out[f"star_{s}_mae"]  = mae(y_true[mask], y_pred[mask])  if mask.sum() else None
    return out

def baseline_predict(df, um, im, gm):
    u = df[COL_USER].map(um).fillna(gm)
    i = df[COL_ITEM].map(im).fillna(gm)
    return ((u + i) / 2.0).clip(1.0, 5.0).values.astype(np.float32)

def to_jsonable(obj):
    if isinstance(obj, dict):          return {str(k): to_jsonable(v) for k,v in obj.items()}
    if isinstance(obj, (list, tuple)): return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):    return obj.tolist()
    if isinstance(obj, np.integer):    return int(obj)
    if isinstance(obj, np.floating):   return float(obj)
    if isinstance(obj, np.bool_):      return bool(obj)
    return obj

# Ground truth + baseline
y_true = test_warm[COL_RATING].astype(np.float32).values
y_base = baseline_predict(test_warm, user_mean_map, item_mean_map, GLOBAL_MEAN)

# train_fit/val_fit (dùng cho group-aware ridge)
train_s = train.sort_values(COL_DATE, na_position="last").reset_index(drop=True)
cut = int(len(train_s) * 0.90)
train_fit_s4 = train_s.iloc[:cut].copy().reset_index(drop=True)
val_fit_s4   = train_s.iloc[cut:].copy().reset_index(drop=True)

print(f"test_warm    : {len(test_warm):,}")
print(f"train_fit_s4 : {len(train_fit_s4):,}")
print(f"val_fit_s4   : {len(val_fit_s4):,}")
print(evaluate(y_true, y_base, "baseline"))

test_warm    : 1,349,731
train_fit_s4 : 5,819,745
val_fit_s4   : 646,639
{'model': 'baseline', 'n': 1349731, 'rmse': 1.1288197626161487, 'mae': 0.8799618482589722}


In [3]:
# ============================================
# CELL 3 — LOAD STAGE 3 OUTPUTS
# ============================================
import pandas as pd
import numpy as np

# 1. Test predictions (Vẫn load từ file .npy vì chúng dành cho tập test)
y_a = np.load(STAGE3_INPUT_DIR / "model_a_preds.npy")
y_b = np.load(STAGE3_INPUT_DIR / "model_b_preds.npy")
y_c = np.load(STAGE3_INPUT_DIR / "model_c_preds.npy")
y_d = np.load(STAGE3_INPUT_DIR / "model_d_preds.npy")

# 2. Đọc file parquet chứa dữ liệu tổng hợp OOF của Stage 3
df_oof = pd.read_parquet(STAGE3_INPUT_DIR / "stage3_oof.parquet")

# Lấy ground truth và baseline đúng tên cột
y_val_true = df_oof[COL_RATING].values
y_val_base = df_oof['baseline'].values 

# Lấy luôn dự đoán OOF của các model trực tiếp từ DataFrame cho chuẩn xác
y_val_a    = df_oof['model_a'].values
y_val_b    = df_oof['model_b'].values
y_val_c    = df_oof['model_c'].values
y_val_d    = df_oof['model_d'].values

# Validation index 
val_index = df_oof[[COL_USER, COL_ITEM]]

# Sanity checks
for name, arr in [("a",y_a),("b",y_b),("c",y_c),("d",y_d)]:
    assert len(arr) == len(test_warm), f"Test pred length mismatch: {name}"
    assert np.isfinite(arr).all(),     f"Non-finite in test pred: {name}"

for name, arr in [("val_base",y_val_base),("val_a",y_val_a),
                   ("val_b",y_val_b),("val_c",y_val_c),("val_d",y_val_d)]:
    assert len(arr) == len(y_val_true), f"OOF length mismatch: {name}"

assert len(val_index) == len(y_val_true), "val_index length mismatch"

print("Test preds  :", y_a.shape, y_b.shape, y_c.shape, y_d.shape)
print("OOF preds   :", y_val_a.shape, y_val_b.shape, y_val_c.shape, y_val_d.shape)
print("val_index   :", val_index.shape)

# Đánh giá các model
results_single = [
    evaluate(y_val_true, y_val_base, "baseline"),
    evaluate(y_val_true, y_val_a,    "model_a"),
    evaluate(y_val_true, y_val_b,    "model_b"),
    evaluate(y_val_true, y_val_c,    "model_c"),
    evaluate(y_val_true, y_val_d,    "model_d"),
]
display(pd.DataFrame(results_single).sort_values("rmse").reset_index(drop=True))

Test preds  : (1349731,) (1349731,) (1349731,) (1349731,)
OOF preds   : (533477,) (533477,) (533477,) (533477,)
val_index   : (533477, 2)


,model,n,rmse,mae
0,model_b,533477,1.084474,0.828581
1,model_d,533477,1.106180,0.832812
2,model_c,533477,1.106182,0.832835
3,baseline,533477,1.106182,0.832835
4,model_a,533477,1.106776,0.832440


In [4]:
# ============================================
# CELL 4 — ENSEMBLE INFRASTRUCTURE
# ============================================
# Pred banks
pred_bank_val  = {"base":y_val_base,"a":y_val_a,"b":y_val_b,"c":y_val_c,"d":y_val_d}
pred_bank_test = {"base":y_base,    "a":y_a,    "b":y_b,    "c":y_c,    "d":y_d}

# OOF meta split 70/30
n_oof = len(y_val_true)
split_meta = int(n_oof * 0.70)
meta_fit_mask    = np.zeros(n_oof, dtype=bool); meta_fit_mask[:split_meta]  = True
meta_select_mask = np.zeros(n_oof, dtype=bool); meta_select_mask[split_meta:] = True
print(f"meta_fit={meta_fit_mask.sum():,} | meta_select={meta_select_mask.sum():,}")

# Candidate registry
compare_rows    = []
candidate_store = {}

def register_candidate(name, family, oof_pred, test_pred,
                        weights=None, params=None, eligible=True):
    oof_pred  = clip_rating(np.asarray(oof_pred,  np.float32))
    test_pred = clip_rating(np.asarray(test_pred, np.float32))

    oof_sel  = evaluate(y_val_true[meta_select_mask], oof_pred[meta_select_mask], name)
    oof_full = evaluate(y_val_true,                   oof_pred,                   name)
    test_res = evaluate(y_true,                       test_pred,                  name)

    record = {
        "model":                  name,
        "family":                 family,
        "eligible_for_selection": bool(eligible),
        "oof_rmse":               float(oof_sel["rmse"]),   # dùng để sort
        "oof_mae":                float(oof_sel["mae"]),
        "oof_full_rmse":          float(oof_full["rmse"]),
        "oof_full_mae":           float(oof_full["mae"]),
        "test_rmse":              float(test_res["rmse"]),
        "test_mae":               float(test_res["mae"]),
        "weights_json":           json.dumps(to_jsonable(weights or {}), sort_keys=True),
        "params_json":            json.dumps(to_jsonable(params  or {}), sort_keys=True),
    }
    compare_rows.append(record)
    candidate_store[name] = {
        "family":    family,
        "oof_pred":  oof_pred,
        "test_pred": test_pred,
        "weights":   to_jsonable(weights or {}),
        "params":    to_jsonable(params  or {}),
        "eligible":  bool(eligible),
    }
    return record

def weighted_blend(preds_dict, weights_dict):
    out = np.zeros(len(next(iter(preds_dict.values()))), dtype=np.float32)
    total_w = 0.0
    for k, w in weights_dict.items():
        w = float(w)
        if w == 0 or k not in preds_dict: continue
        out    += w * np.asarray(preds_dict[k], np.float32)
        total_w += w
    return clip_rating(out / max(total_w, 1e-9))

def simplex_weights(keys, step):
    """Generator: tất cả convex combinations với bước step."""
    total = int(round(1.0 / step))
    for units in itertools.product(range(total+1), repeat=len(keys)):
        if sum(units) != total: continue
        yield {k: float(u * step) for k, u in zip(keys, units)}

def grid_blend_best(name, keys, step):
    best_r, best_w, best_oof, best_test = np.inf, None, None, None
    for w in simplex_weights(keys, step):
        oof_p  = weighted_blend({k: pred_bank_val[k]  for k in keys}, w)
        sel_r  = rmse(y_val_true[meta_select_mask], oof_p[meta_select_mask])
        if sel_r < best_r:
            best_r    = sel_r
            best_w    = w
            best_oof  = oof_p
            best_test = weighted_blend({k: pred_bank_test[k] for k in keys}, w)
    return best_w, best_oof, best_test

def build_meta_features(base, a, b, c, d):
    """12 meta-features: diffs, abs-diffs, mean, std, range, base."""
    base,a,b,c,d = [np.asarray(x,np.float32) for x in [base,a,b,c,d]]
    stack = np.column_stack([a,b,c,d])
    X = np.column_stack([
        b-base, c-base, d-base, a-base,
        np.abs(b-base), np.abs(c-base), np.abs(d-base), np.abs(a-base),
        stack.mean(1), stack.std(1),
        stack.max(1)-stack.min(1),
        base,
    ]).astype(np.float32)
    names = ["b-base","c-base","d-base","a-base",
             "|b-base|","|c-base|","|d-base|","|a-base|",
             "mean","std","range","base"]
    return X, names

print("Ensemble infrastructure ready.")

meta_fit=373,433 | meta_select=160,044
Ensemble infrastructure ready.


In [5]:
# ============================================
# CELL 5 — GROUP 1: SINGLE MODELS
# ============================================
print("="*60)
print("GROUP 1 — Single Models")
print("="*60)

for name, (ov, ot) in {
    "baseline": (y_val_base, y_base),
    "model_a":  (y_val_a,    y_a),
    "model_b":  (y_val_b,    y_b),
    "model_c":  (y_val_c,    y_c),
    "model_d":  (y_val_d,    y_d),
}.items():
    r = register_candidate(name, "single_model", ov, ot,
                           weights={name:1.0}, params={"kind":"single_model"})
    print(f"  {name:<12} oof_rmse={r['oof_rmse']:.5f}  test_rmse={r['test_rmse']:.5f}  test_mae={r['test_mae']:.5f}")

display(pd.DataFrame(compare_rows).sort_values("oof_rmse").reset_index(drop=True))

GROUP 1 — Single Models
  baseline     oof_rmse=1.12525  test_rmse=1.12882  test_mae=0.87996
  model_a      oof_rmse=1.12597  test_rmse=1.12938  test_mae=0.87960
  model_b      oof_rmse=1.10315  test_rmse=1.12059  test_mae=0.85073
  model_c      oof_rmse=1.12525  test_rmse=1.12882  test_mae=0.87996
  model_d      oof_rmse=1.12525  test_rmse=1.12882  test_mae=0.87994


,model,family,eligible_for_selection,oof_rmse,oof_mae,oof_full_rmse,oof_full_mae,test_rmse,test_mae,weights_json,params_json
0,model_b,single_model,True,1.103151,0.843250,1.084474,0.828581,1.120590,0.850729,"{""model_b"": 1.0}","{""kind"": ""single_model""}"
1,model_d,single_model,True,1.125247,0.848284,1.106180,0.832812,1.128820,0.879938,"{""model_d"": 1.0}","{""kind"": ""single_model""}"
2,model_c,single_model,True,1.125250,0.848313,1.106182,0.832835,1.128820,0.879962,"{""model_c"": 1.0}","{""kind"": ""single_model""}"
3,baseline,single_model,True,1.125250,0.848313,1.106182,0.832835,1.128820,0.879962,"{""baseline"": 1.0}","{""kind"": ""single_model""}"
4,model_a,single_model,True,1.125971,0.848163,1.106776,0.832440,1.129385,0.879598,"{""model_a"": 1.0}","{""kind"": ""single_model""}"


In [6]:
# ============================================
# CELL 6 — GROUP 2: MANUAL BLENDS
# ============================================
print("="*60)
print("GROUP 2 — Manual Blends")
print("="*60)

manual_cfgs = {
    "blend_base50_b50":         {"base":0.50,"b":0.50},
    "blend_base55_b35_c10":     {"base":0.55,"b":0.35,"c":0.10},
    "blend_base55_b30_c10_d05": {"base":0.55,"b":0.30,"c":0.10,"d":0.05},
}
for name, w in manual_cfgs.items():
    oof_p  = weighted_blend(pred_bank_val,  w)
    test_p = weighted_blend(pred_bank_test, w)
    r = register_candidate(name, "manual_blend", oof_p, test_p,
                           weights=w, params={"kind":"manual"})
    print(f"  {name:<35} oof={r['oof_rmse']:.5f}  test={r['test_rmse']:.5f}  MAE={r['test_mae']:.5f}")

GROUP 2 — Manual Blends
  blend_base50_b50                    oof=1.10408  test=1.12138  MAE=0.86431
  blend_base55_b35_c10                oof=1.10833  test=1.12292  MAE=0.86876
  blend_base55_b30_c10_d05            oof=1.11015  test=1.12356  MAE=0.87028


In [7]:
# ============================================
# CELL 7 — GROUP 3: GRID BLENDS
# ============================================
print("="*60)
print("GROUP 3 — Grid Blends")
print("="*60)

for name, keys, step in [
    ("grid_base_b",     ("base","b"),           0.05),
    ("grid_base_b_c",   ("base","b","c"),        0.05),
    ("grid_base_b_c_d", ("base","b","c","d"),    0.10),
]:
    print(f"  Searching {name} (step={step})...")
    best_w, oof_p, test_p = grid_blend_best(name, keys, step)
    r = register_candidate(name, "grid_blend", oof_p, test_p,
                           weights=best_w, params={"keys":list(keys),"step":step})
    print(f"  {name:<25} weights={best_w}  oof={r['oof_rmse']:.5f}  test={r['test_rmse']:.5f}")

display(pd.DataFrame(compare_rows).sort_values(
    ["eligible_for_selection","oof_rmse"], ascending=[False,True]
).reset_index(drop=True))

GROUP 3 — Grid Blends
  Searching grid_base_b (step=0.05)...
  grid_base_b               weights={'base': 0.25, 'b': 0.75}  oof=1.10105  test=1.12015
  Searching grid_base_b_c (step=0.05)...
  grid_base_b_c             weights={'base': 0.0, 'b': 0.75, 'c': 0.25}  oof=1.10105  test=1.12015
  Searching grid_base_b_c_d (step=0.1)...
  grid_base_b_c_d           weights={'base': 0.0, 'b': 0.8, 'c': 0.0, 'd': 0.2}  oof=1.10106  test=1.12010


,model,family,eligible_for_selection,oof_rmse,oof_mae,oof_full_rmse,oof_full_mae,test_rmse,test_mae,weights_json,params_json
0,grid_base_b,grid_blend,True,1.101053,0.841744,1.082373,0.826975,1.120150,0.857295,"{""b"": 0.75, ""base"": 0.25}","{""keys"": [""base"", ""b""], ""step"": 0.05}"
1,grid_base_b_c,grid_blend,True,1.101053,0.841744,1.082373,0.826975,1.120150,0.857295,"{""b"": 0.75, ""base"": 0.0, ""c"": 0.25}","{""keys"": [""base"", ""b"", ""c""], ""step"": 0.05}"
2,grid_base_b_c_d,grid_blend,True,1.101062,0.841860,1.082388,0.827116,1.120104,0.855943,"{""b"": 0.8, ""base"": 0.0, ""c"": 0.0, ""d"": 0.2}","{""keys"": [""base"", ""b"", ""c"", ""d""], ""step"": 0.1}"
3,model_b,single_model,True,1.103151,0.843250,1.084474,0.828581,1.120590,0.850729,"{""model_b"": 1.0}","{""kind"": ""single_model""}"
4,blend_base50_b50,manual_blend,True,1.104075,0.842300,1.085330,0.827336,1.121379,0.864310,"{""b"": 0.5, ""base"": 0.5}","{""kind"": ""manual""}"
5,blend_base55_b35_c10,manual_blend,True,1.108333,0.843418,1.089516,0.828323,1.122917,0.868756,"{""b"": 0.35, ""base"": 0.55, ""c"": 0.1}","{""kind"": ""manual""}"
6,blend_base55_b30_c10_d05,manual_blend,True,1.110155,0.843917,1.091310,0.828778,1.123562,0.870280,"{""b"": 0.3, ""base"": 0.55, ""c"": 0.1, ""d"": 0.05}","{""kind"": ""manual""}"
7,model_d,single_model,True,1.125247,0.848284,1.106180,0.832812,1.128820,0.879938,"{""model_d"": 1.0}","{""kind"": ""single_model""}"
8,baseline,single_model,True,1.125250,0.848313,1.106182,0.832835,1.128820,0.879962,"{""baseline"": 1.0}","{""kind"": ""single_model""}"
9,model_c,single_model,True,1.125250,0.848313,1.106182,0.832835,1.128820,0.879962,"{""model_c"": 1.0}","{""kind"": ""single_model""}"


In [8]:
# ============================================
# CELL 8 — GROUP 4: RESIDUAL BLEND
# ============================================
print("="*60)
print("GROUP 4 — Residual Blend")
print("="*60)

base_v = pred_bank_val["base"];  base_t = pred_bank_test["base"]
best_res_r, best_wb, best_wc, best_wd = np.inf, 0.0, 0.0, 0.0

# Search trên meta_fit
vals = np.arange(0, 2.01, 0.10)
for wb in vals:
    for wc in vals:
        for wd in vals:
            oof_p = clip_rating(
                base_v + wb*(pred_bank_val["b"]-base_v)
                       + wc*(pred_bank_val["c"]-base_v)
                       + wd*(pred_bank_val["d"]-base_v)
            )
            r = rmse(y_val_true[meta_select_mask], oof_p[meta_select_mask])
            if r < best_res_r:
                best_res_r = r
                best_wb, best_wc, best_wd = wb, wc, wd

oof_res  = clip_rating(base_v + best_wb*(pred_bank_val["b"] -base_v)
                               + best_wc*(pred_bank_val["c"] -base_v)
                               + best_wd*(pred_bank_val["d"] -base_v))
test_res = clip_rating(base_t + best_wb*(pred_bank_test["b"]-base_t)
                               + best_wc*(pred_bank_test["c"]-base_t)
                               + best_wd*(pred_bank_test["d"]-base_t))

r = register_candidate(
    "residual_blend_base_b_c_d", "residual_blend", oof_res, test_res,
    weights={"w_b":best_wb,"w_c":best_wc,"w_d":best_wd},
    params={"formula":"base+wB*(B-base)+wC*(C-base)+wD*(D-base)"}
)
print(f"  Best wB={best_wb:.2f} wC={best_wc:.2f} wD={best_wd:.2f}")
print(f"  oof={r['oof_rmse']:.5f}  test={r['test_rmse']:.5f}  MAE={r['test_mae']:.5f}")

GROUP 4 — Residual Blend
  Best wB=0.80 wC=0.00 wD=1.10
  oof=1.10106  test=1.12011  MAE=0.85592


In [9]:
# ============================================
# CELL 9 — GROUP 5: RIDGE POSITIVE (NNLS)
# ============================================
print("="*60)
print("GROUP 5 — Ridge Positive (base + B + C + D)")
print("="*60)

X_val_4  = np.column_stack([y_val_base, y_val_b, y_val_c, y_val_d]).astype(np.float32)
X_test_4 = np.column_stack([y_base,     y_b,     y_c,     y_d    ]).astype(np.float32)

ridge_pos = Ridge(alpha=1.0, positive=True, fit_intercept=False)
ridge_pos.fit(X_val_4[meta_fit_mask], y_val_true[meta_fit_mask])

coef     = ridge_pos.coef_.astype(np.float32)
coef_sum = coef.sum()
coef_n   = coef / max(coef_sum, 1e-9)

oof_ridge  = clip_rating(ridge_pos.predict(X_val_4))
test_ridge = clip_rating(ridge_pos.predict(X_test_4))

ridge_weights = {
    "base": float(coef[0]), "b": float(coef[1]),
    "c":    float(coef[2]), "d": float(coef[3]),
    "norm_base": float(coef_n[0]), "norm_b": float(coef_n[1]),
    "norm_c":    float(coef_n[2]), "norm_d": float(coef_n[3]),
}
r = register_candidate(
    "ridge_positive_base_b_c_d", "ridge_positive",
    oof_ridge, test_ridge, weights=ridge_weights,
    params={"alpha":1.0,"positive":True,"fit_intercept":False,
            "features":["base","b","c","d"]}
)
print(f"  Coef (raw)   : base={coef[0]:.4f}  b={coef[1]:.4f}  c={coef[2]:.4f}  d={coef[3]:.4f}")
print(f"  Coef (norm)  : base={coef_n[0]:.3f}  b={coef_n[1]:.3f}  c={coef_n[2]:.3f}  d={coef_n[3]:.3f}")
print(f"  oof={r['oof_rmse']:.5f}  test={r['test_rmse']:.5f}  MAE={r['test_mae']:.5f}")

GROUP 5 — Ridge Positive (base + B + C + D)
  Coef (raw)   : base=0.0680  b=0.7705  c=0.0680  d=0.0935
  Coef (norm)  : base=0.068  b=0.771  c=0.068  d=0.094
  oof=1.10104  test=1.12013  MAE=0.85668


In [10]:
# ============================================
# CELL 10 — GROUP 6+7: META-FEATURE MODELS
# ============================================
print("="*60)
print("GROUP 6+7 — Meta-feature Ridge / ElasticNet / HGBR")
print("="*60)

X_val_meta,  meta_names = build_meta_features(y_val_base,y_val_a,y_val_b,y_val_c,y_val_d)
X_test_meta, _          = build_meta_features(y_base,    y_a,    y_b,    y_c,    y_d)
y_res_val = (y_val_true - y_val_base).astype(np.float32)

# ── 6a. Residual Ridge Meta ──
print("\n[6a] Residual Ridge Meta...")
best_r_meta, best_alpha_rm = np.inf, 1.0
for alpha in [0.01, 0.1, 1.0, 5.0, 10.0]:
    m = Ridge(alpha=alpha, fit_intercept=True)
    m.fit(X_val_meta[meta_fit_mask], y_res_val[meta_fit_mask])
    oof_p = clip_rating(y_val_base + m.predict(X_val_meta))
    r_sel = rmse(y_val_true[meta_select_mask], oof_p[meta_select_mask])
    print(f"  alpha={alpha:<6} sel_rmse={r_sel:.5f}")
    if r_sel < best_r_meta:
        best_r_meta, best_alpha_rm = r_sel, alpha

rm_final = Ridge(alpha=best_alpha_rm, fit_intercept=True)
rm_final.fit(X_val_meta, y_res_val)
oof_rm  = clip_rating(y_val_base + rm_final.predict(X_val_meta))
test_rm = clip_rating(y_base     + rm_final.predict(X_test_meta))
r = register_candidate("residual_ridge_meta","meta_ridge", oof_rm, test_rm,
    weights={"intercept":float(rm_final.intercept_),
             "coef":{n:float(c) for n,c in zip(meta_names, rm_final.coef_)}},
    params={"alpha":best_alpha_rm,"feature_names":meta_names})
print(f"  => oof={r['oof_rmse']:.5f}  test={r['test_rmse']:.5f}  MAE={r['test_mae']:.5f}")

# ── 6b. ElasticNet Residual Meta ──
print("\n[6b] ElasticNet Residual Meta...")
try:
    best_r_en, best_en_cfg = np.inf, {"alpha":0.001,"l1_ratio":0.5}
    for alpha in [0.0001, 0.001, 0.01]:
        for l1r in [0.1, 0.5, 0.9]:
            m = ElasticNet(alpha=alpha, l1_ratio=l1r, fit_intercept=True,
                           max_iter=3000, tol=1e-3, random_state=42)
            m.fit(X_val_meta[meta_fit_mask], y_res_val[meta_fit_mask])
            oof_p = clip_rating(y_val_base + m.predict(X_val_meta))
            r_sel = rmse(y_val_true[meta_select_mask], oof_p[meta_select_mask])
            print(f"  alpha={alpha}  l1={l1r}  sel_rmse={r_sel:.5f}")
            if r_sel < best_r_en:
                best_r_en, best_en_cfg = r_sel, {"alpha":alpha,"l1_ratio":l1r}

    en_final = ElasticNet(**best_en_cfg, fit_intercept=True,
                           max_iter=3000, tol=1e-3, random_state=42)
    en_final.fit(X_val_meta, y_res_val)
    oof_en  = clip_rating(y_val_base + en_final.predict(X_val_meta))
    test_en = clip_rating(y_base     + en_final.predict(X_test_meta))
    r = register_candidate("elasticnet_residual_meta","meta_elasticnet",
        oof_en, test_en,
        weights={"intercept":float(en_final.intercept_),
                 "coef":{n:float(c) for n,c in zip(meta_names,en_final.coef_)}},
        params={**best_en_cfg,"feature_names":meta_names})
    print(f"  => oof={r['oof_rmse']:.5f}  test={r['test_rmse']:.5f}  MAE={r['test_mae']:.5f}")
except Exception as e:
    print(f"  Skipped ElasticNet: {e}")

# ── 7. HistGradientBoosting Residual ──
print("\n[7] HistGradientBoosting Residual Meta...")
try:
    rng = np.random.RandomState(42)
    fit_idx = np.where(meta_fit_mask)[0]
    if len(fit_idx) > 300_000:
        fit_idx = rng.choice(fit_idx, 300_000, replace=False)

    best_r_hg, best_hg_cfg = np.inf, None
    hg_results = []

    for max_iter in [100, 200]:
        for lr in [0.03, 0.05]:
            for leaves in [15, 31]:
                for l2 in [0.0, 0.1]:
                    m = HistGradientBoostingRegressor(
                        max_iter=max_iter, learning_rate=lr,
                        max_leaf_nodes=leaves, l2_regularization=l2,
                        random_state=42
                    )
                    m.fit(X_val_meta[fit_idx], y_res_val[fit_idx])
                    oof_p = clip_rating(y_val_base + m.predict(X_val_meta))
                    r_sel = rmse(y_val_true[meta_select_mask], oof_p[meta_select_mask])
                    hg_results.append({"max_iter":max_iter,"lr":lr,
                                        "leaves":leaves,"l2":l2,"sel_rmse":r_sel})
                    print(f"  iter={max_iter} lr={lr} leaves={leaves} l2={l2} => {r_sel:.5f}")
                    if r_sel < best_r_hg:
                        best_r_hg  = r_sel
                        best_hg_cfg = {"max_iter":max_iter,"learning_rate":lr,
                                       "max_leaf_nodes":leaves,"l2_regularization":l2}

    # Refit on full OOF (sampled nếu quá lớn)
    full_idx = np.arange(len(X_val_meta))
    if len(full_idx) > 500_000:
        full_idx = rng.choice(full_idx, 500_000, replace=False)

    hg_final = HistGradientBoostingRegressor(**best_hg_cfg, random_state=42)
    hg_final.fit(X_val_meta[full_idx], y_res_val[full_idx])
    oof_hg  = clip_rating(y_val_base + hg_final.predict(X_val_meta))
    test_hg = clip_rating(y_base     + hg_final.predict(X_test_meta))
    r = register_candidate("hist_gradient_boosting_residual","meta_hgbr",
        oof_hg, test_hg,
        weights={"model_type":"HistGradientBoostingRegressor"},
        params={**best_hg_cfg,"feature_names":meta_names})
    print(f"  => oof={r['oof_rmse']:.5f}  test={r['test_rmse']:.5f}  MAE={r['test_mae']:.5f}")
except Exception as e:
    print(f"  Skipped HGBR: {e}")

GROUP 6+7 — Meta-feature Ridge / ElasticNet / HGBR

[6a] Residual Ridge Meta...
  alpha=0.01   sel_rmse=1.10069
  alpha=0.1    sel_rmse=1.10069
  alpha=1.0    sel_rmse=1.10069
  alpha=5.0    sel_rmse=1.10069
  alpha=10.0   sel_rmse=1.10069
  => oof=1.10055  test=1.11966  MAE=0.86059

[6b] ElasticNet Residual Meta...
  alpha=0.0001  l1=0.1  sel_rmse=1.10068
  alpha=0.0001  l1=0.5  sel_rmse=1.10068
  alpha=0.0001  l1=0.9  sel_rmse=1.10068
  alpha=0.001  l1=0.1  sel_rmse=1.10067
  alpha=0.001  l1=0.5  sel_rmse=1.10067
  alpha=0.001  l1=0.9  sel_rmse=1.10068
  alpha=0.01  l1=0.1  sel_rmse=1.10101
  alpha=0.01  l1=0.5  sel_rmse=1.10122
  alpha=0.01  l1=0.9  sel_rmse=1.10157
  => oof=1.10054  test=1.11972  MAE=0.86105

[7] HistGradientBoosting Residual Meta...
  iter=100 lr=0.03 leaves=15 l2=0.0 => 1.10034
  iter=100 lr=0.03 leaves=15 l2=0.1 => 1.10034
  iter=100 lr=0.03 leaves=31 l2=0.0 => 1.10041
  iter=100 lr=0.03 leaves=31 l2=0.1 => 1.10041
  iter=100 lr=0.05 leaves=15 l2=0.0 => 1.10028


In [11]:
# ============================================
# CELL 11 — GROUP 8: GROUP-AWARE RIDGE (diagnostic)
# ============================================
print("="*60)
print("GROUP 8 — Group-Aware Ridge (NOT eligible)")
print("="*60)

def group_label(user_cnt, item_cnt):
    u = np.where(user_cnt<=2,"low", np.where(user_cnt<=10,"mid","high"))
    i = np.where(item_cnt<=2,"low", np.where(item_cnt<=20,"mid","high"))
    return np.char.add(np.char.add(u.astype(str),"_"), i.astype(str))

uc_val  = val_index[COL_USER].map(train_fit_s4.groupby(COL_USER).size()).fillna(0).astype(int).values
ic_val  = val_index[COL_ITEM].map(train_fit_s4.groupby(COL_ITEM).size()).fillna(0).astype(int).values
uc_test = test_warm[COL_USER].map(train.groupby(COL_USER).size()).fillna(0).astype(int).values
ic_test = test_warm[COL_ITEM].map(train.groupby(COL_ITEM).size()).fillna(0).astype(int).values

val_groups  = group_label(uc_val,  ic_val)
test_groups = group_label(uc_test, ic_test)

oof_ga  = clip_rating(oof_ridge.copy())
test_ga = clip_rating(test_ridge.copy())

for grp in sorted(set(val_groups)):
    vm = val_groups == grp
    tm = test_groups == grp
    if vm.sum() < 5000:
        continue
    try:
        m = Ridge(alpha=1.0, positive=True, fit_intercept=False)
        m.fit(X_val_4[vm], y_val_true[vm])
        oof_ga[vm]  = clip_rating(m.predict(X_val_4[vm]))
        if tm.sum() > 0:
            test_ga[tm] = clip_rating(m.predict(X_test_4[tm]))
        print(f"  Group {grp:<15}: n_val={vm.sum():,}  n_test={tm.sum():,}")
    except Exception as e:
        print(f"  Group {grp}: fallback ({e})")

r = register_candidate(
    "group_aware_ridge", "group_aware_ridge",
    oof_ga, test_ga,
    weights={"fallback":"ridge_positive_base_b_c_d"},
    params={"alpha":1.0,"min_group_rows":5000,
            "note":"NOT eligible — fits/evaluates on same OOF groups"},
    eligible=False    # ← không dùng để select
)
print(f"  oof={r['oof_rmse']:.5f}  test={r['test_rmse']:.5f}  MAE={r['test_mae']:.5f}  [diagnostic only]")

GROUP 8 — Group-Aware Ridge (NOT eligible)
  Group high_high      : n_val=58,120  n_test=165,168
  Group high_low       : n_val=6,274  n_test=14,167
  Group high_mid       : n_val=35,630  n_test=82,355
  Group low_high       : n_val=80,545  n_test=194,111
  Group low_low        : n_val=6,248  n_test=12,667
  Group low_mid        : n_val=41,925  n_test=83,019
  Group mid_high       : n_val=185,493  n_test=521,592
  Group mid_low        : n_val=16,981  n_test=38,377
  Group mid_mid        : n_val=102,261  n_test=238,275
  oof=1.10069  test=1.12116  MAE=0.86313  [diagnostic only]


In [12]:
# ============================================
# CELL 12 — COMPARE ALL + SELECT BEST
# ============================================
print("="*70)
print("ALL CANDIDATES — sorted by oof_rmse (eligible first)")
print("="*70)

compare_df = (
    pd.DataFrame(compare_rows)
    .sort_values(["eligible_for_selection","oof_rmse","oof_mae","model"],
                  ascending=[False,True,True,True])
    .reset_index(drop=True)
)
display(compare_df[[
    "model","family","eligible_for_selection",
    "oof_rmse","oof_mae","oof_full_rmse",
    "test_rmse","test_mae"
]])

# Select best eligible
eligible_df = compare_df[compare_df["eligible_for_selection"]].reset_index(drop=True)
assert len(eligible_df) > 0, "No eligible candidates!"

best_name      = eligible_df.iloc[0]["model"]
best_cand      = candidate_store[best_name]
best_pred      = best_cand["test_pred"]
best_oof_pred  = best_cand["oof_pred"]

res_best_test  = evaluate(y_true,                        best_pred,     best_name)
res_best_oof_f = evaluate(y_val_true,                    best_oof_pred, best_name)
res_best_oof_s = evaluate(y_val_true[meta_select_mask],
                           best_oof_pred[meta_select_mask], best_name)
res_baseline   = evaluate(y_true, y_base, "baseline")

print(f"\n✓  SELECTED: {best_name}")
print(f"\n{'─'*55}")
print(f"{'':5} {'Model':<30} {'RMSE':>9} {'MAE':>9}")
print(f"{'─'*55}")
print(f"{'TEST':<5} {'Baseline':<30} {res_baseline['rmse']:>9.5f} {res_baseline['mae']:>9.5f}")
print(f"{'TEST':<5} {best_name:<30} {res_best_test['rmse']:>9.5f} {res_best_test['mae']:>9.5f}")
delta_rmse = res_best_test["rmse"] - res_baseline["rmse"]
delta_mae  = res_best_test["mae"]  - res_baseline["mae"]
print(f"{'':5} {'Δ vs baseline':<30} {delta_rmse:>+9.5f} {delta_mae:>+9.5f}")
print(f"{'─'*55}")

# Per-star
print("\nPer-star (Final):")
per_star_base = evaluate_per_star(y_true, y_base)
per_star_best = evaluate_per_star(y_true, best_pred)
ps_rows = []
for s in range(1, 6):
    n    = per_star_best[f"star_{s}_n"]
    b_r  = per_star_base[f"star_{s}_rmse"]
    f_r  = per_star_best[f"star_{s}_rmse"]
    f_m  = per_star_best[f"star_{s}_mae"]
    ps_rows.append({"star":s, "n":n,
                    "baseline_rmse": round(b_r,4) if b_r else None,
                    "final_rmse":    round(f_r,4) if f_r else None,
                    "final_mae":     round(f_m,4) if f_m else None})
display(pd.DataFrame(ps_rows))

# Progression table
print("\nProgression:")
model_b_res = evaluate(y_true, y_b, "model_b_stage3")
prog = [
    ("Stage 0", "Baseline",        res_baseline["rmse"],  res_baseline["mae"]),
    ("Stage 3", "Model B (MF-SGD)",model_b_res["rmse"],   model_b_res["mae"]),
    ("Stage 4", best_name,          res_best_test["rmse"], res_best_test["mae"]),
]
prog_df = pd.DataFrame(prog, columns=["Stage","Model","RMSE","MAE"])
prog_df["ΔRMSE_vs_baseline"] = prog_df["RMSE"].apply(
    lambda r: f"{(r - res_baseline['rmse'])/res_baseline['rmse']*100:+.3f}%"
)
display(prog_df)

ALL CANDIDATES — sorted by oof_rmse (eligible first)


,model,family,eligible_for_selection,oof_rmse,oof_mae,oof_full_rmse,test_rmse,test_mae
0,hist_gradient_boosting_residual,meta_hgbr,True,1.099070,0.843029,1.080260,1.118984,0.863294
1,elasticnet_residual_meta,meta_elasticnet,True,1.100544,0.842196,1.081893,1.119724,0.861047
2,residual_ridge_meta,meta_ridge,True,1.100546,0.841992,1.081878,1.119658,0.860585
3,ridge_positive_base_b_c_d,ridge_positive,True,1.101035,0.841721,1.082356,1.120129,0.856682
4,grid_base_b,grid_blend,True,1.101053,0.841744,1.082373,1.120150,0.857295
5,grid_base_b_c,grid_blend,True,1.101053,0.841744,1.082373,1.120150,0.857295
6,residual_blend_base_b_c_d,residual_blend,True,1.101060,0.841837,1.082388,1.120107,0.855920
7,grid_base_b_c_d,grid_blend,True,1.101062,0.841860,1.082388,1.120104,0.855943
8,model_b,single_model,True,1.103151,0.843250,1.084474,1.120590,0.850729
9,blend_base50_b50,manual_blend,True,1.104075,0.842300,1.085330,1.121379,0.864310



✓  SELECTED: hist_gradient_boosting_residual

───────────────────────────────────────────────────────
      Model                               RMSE       MAE
───────────────────────────────────────────────────────
TEST  Baseline                         1.12882   0.87996
TEST  hist_gradient_boosting_residual   1.11898   0.86329
      Δ vs baseline                   -0.00984  -0.01667
───────────────────────────────────────────────────────

Per-star (Final):


,star,n,baseline_rmse,final_rmse,final_mae
0,1,73663,3.1308,3.0649,3.0391
1,2,77335,2.1588,2.1108,2.0806
2,3,136137,1.2019,1.1847,1.1416
3,4,235084,0.2966,0.3612,0.3059
4,5,827512,0.7125,0.7224,0.6684



Progression:


,Stage,Model,RMSE,MAE,ΔRMSE_vs_baseline
0,Stage 0,Baseline,1.128820,0.879962,+0.000%
1,Stage 3,Model B (MF-SGD),1.120590,0.850729,-0.729%
2,Stage 4,hist_gradient_boosting_residual,1.118984,0.863294,-0.871%


In [13]:
# ============================================
# CELL 13 — SAVE OUTPUTS
# ============================================
print("="*60)
print("Saving Stage 4 artifacts...")
print("="*60)

# Final predictions
np.save(STAGE4_DIR / "stage4_best_preds.npy", best_pred)
compare_df.to_csv(STAGE4_DIR / "stage4_compare.csv", index=False)

# Best result JSON
stage4_result = {
    "selected_model":    best_name,
    "selected_family":   best_cand["family"],
    "selected_by":       "oof_meta_select_rmse_then_mae",
    "oof_full_metrics":  to_jsonable(res_best_oof_f),
    "oof_sel_metrics":   to_jsonable(res_best_oof_s),
    "test_metrics":      to_jsonable(res_best_test),
    "baseline_metrics":  to_jsonable(res_baseline),
    "delta_rmse":        float(delta_rmse),
    "delta_mae":         float(delta_mae),
    "weights":           to_jsonable(best_cand["weights"]),
    "params":            to_jsonable(best_cand["params"]),
    "per_star_baseline": to_jsonable(per_star_base),
    "per_star_final":    to_jsonable(per_star_best),
    "meta_split":        {"fit_rows":int(meta_fit_mask.sum()),
                          "select_rows":int(meta_select_mask.sum()),
                          "fit_ratio":0.70},
    "note": "test_warm used only for final evaluation, not selection",
}
with open(STAGE4_DIR / "stage4_best_result.json", "w") as f:
    json.dump(to_jsonable(stage4_result), f, indent=2)

# All weights
stage4_weights = {
    "selected_model":   best_name,
    "all_candidates": {
        name: {"weights": info["weights"], "params": info["params"],
               "eligible": info["eligible"]}
        for name, info in candidate_store.items()
    }
}
with open(STAGE4_DIR / "stage4_weights.json", "w") as f:
    json.dump(to_jsonable(stage4_weights), f, indent=2)

print(f"\nSaved to {STAGE4_DIR}:")
for p in sorted(STAGE4_DIR.iterdir()):
    print(f"  {p.name:<40} {p.stat().st_size/1e6:.2f} MB")

print("\n=== STAGE 4 DONE ===")

Saving Stage 4 artifacts...

Saved to /kaggle/working/stage4_artifacts:
  stage4_best_preds.npy                    5.40 MB
  stage4_best_result.json                  0.00 MB
  stage4_compare.csv                       0.01 MB
  stage4_weights.json                      0.01 MB

=== STAGE 4 DONE ===
